# ZINC Streaming Demo

This notebook showcases the schema-frozen streaming training path for `ConditionalNodeFieldGraphGenerator`.

- source: raw ZINC CSV
- warmup: first 1000 accepted graphs
- stream limit: `0.1`
- targets: none
- final cells: generate 7 graphs without feasibility filtering, then 7 with filtering


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from pathlib import Path
import os
import random

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')

import numpy as np
from IPython.core.display import HTML

HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell; text-align: center; vertical-align: middle;}</style>')

from conditional_node_field_graph_generator.notebooks import configure_notebook, download_zinc_dataset
globals().update(configure_notebook(require_nsppk=True, print_torch=True))

from abstractgraph_graphicalizer.chem import draw_molecules
from conditional_node_field_graph_generator.extensions.demo import show_molecules
from conditional_node_field_graph_generator.extensions.demo.pipeline import build_graph_generator


In [ ]:
RANDOM_SEED = 7
STREAM_LIMIT = 0.1
WARMUP_SIZE = 1024
STREAM_BATCH_SIZE = 32
MAXIMUM_EPOCHS = 100
EMBEDDING_DIM = 64
MODEL_NAME = f'zinc-streaming-n{EMBEDDING_DIM}-s{STREAM_LIMIT}-w{WARMUP_SIZE}-b{STREAM_BATCH_SIZE}-e{MAXIMUM_EPOCHS}'
ZINC_DATA_ROOT = NOTEBOOK_DATA_ROOT / 'zinc'
ZINC_FILENAME = 'zinc_18.csv'
DECODER_N_JOBS = 1
STREAM_SNAPSHOT_EVERY_N_BATCHES = 40
URI = ZINC_DATA_ROOT / "zinc18.csv"
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


In [ ]:
graph_generator = build_graph_generator(
    latent_embedding_dimension=EMBEDDING_DIM,
    number_of_transformer_layers=2,
    transformer_attention_head_count=4,
    maximum_epochs=MAXIMUM_EPOCHS,
    batch_size=STREAM_BATCH_SIZE,
    verbose=1,
    decoder_n_jobs=DECODER_N_JOBS,
    artifact_root=ARTIFACT_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    model_name=MODEL_NAME,
    model_dir=SAVED_GENERATOR_ROOT,
    stream_snapshot_every_n_batches=STREAM_SNAPSHOT_EVERY_N_BATCHES,
)
graph_generator.graph_decoder.diagnostic_graph_renderer = draw_molecules


In [ ]:


graph_generator.fit_from_stream(
    URI,
    "zinc_csv",
    warmup_size=WARMUP_SIZE,
    batch_size=STREAM_BATCH_SIZE,
    limit=STREAM_LIMIT,
    random_state=RANDOM_SEED,
    verbose=True,
)

print('stream_seen_ =', graph_generator.stream_seen_)
print('stream_warmup_count_ =', graph_generator.stream_warmup_count_)
print('stream_training_seen_ =', graph_generator.stream_training_seen_)
print('stream_training_accepted_ =', graph_generator.stream_training_accepted_)
print('stream_training_skipped_ =', graph_generator.stream_training_skipped_)
print('stream_acceptance_rate_ =', graph_generator.stream_acceptance_rate_)


In [ ]:
raw_samples = graph_generator.sample(
    n_samples=7,
    apply_feasibility_filtering=False,
)
show_molecules(raw_samples, n=7, title='Streaming ZINC samples without feasibility filtering')


In [ ]:
if graph_generator.feasibility_estimator is None:
    raise RuntimeError('Feasibility estimator is unavailable in this environment.')

filtered_samples = graph_generator.sample(
    n_samples=7,
    apply_feasibility_filtering=True,
)
show_molecules(filtered_samples, n=7, title='Streaming ZINC samples with feasibility filtering')
